# 🏮 AI기반 지역관광 맞춤 여행 플래너 — 전통시장 DB 통합판 (Colab)

기존에 보내주신 `korea-trip-basket-colab.ipynb`의 구조를 유지하면서, **전통시장 DB(SQLite, 23개 테이블)** 와
**사용자 / 상인 / 관리자 3분할 화면**을 결합한 버전입니다. PRD v18의 FR 번호를 코드 주석에 그대로 표기해두었습니다.

## 실행 순서
1. **셀 1** 실행 → 패키지 설치
2. 왼쪽 파일 탭에 `전통시장_상가매칭_광주전남_FULL.csv` 업로드 (이미 만들어둔 `travel_planner.db`가 있다면 그것을 업로드해도 됩니다)
3. **셀 2~4** 순서대로 실행 → DB 스키마 생성 + CSV 자동 적재 + 로그인/역할별 화면 로직 정의
4. **셀 5** 실행 → 공유 링크(`https://xxxx.gradio.live`) 생성, 클릭해서 접속

## 이번 버전에서 달라진 점
- **화면이 3분할**됩니다: `🔐 로그인/회원가입` / (일반 사용자 화면) / `🧑‍🌾 상인 대시보드` / `🛠️ 관리자 대시보드`
- **일반 사용자**는 상인·관리자 화면 탭이 아예 보이지 않습니다. 로그인 성공 시에만 자신의 역할에 맞는 탭이 나타납니다.
- **관리자**는 별도 회원가입이 없고(PRD FR-011: 초대 기반), 데모 편의를 위해 **"🔑 관리자 계정으로 바로 입장" 버튼** 하나로 즉시 관리자 화면에 들어갈 수 있습니다. (실서비스에서는 2단계 인증(FR-010)으로 대체)
- **상단에 로그인 상태 표시 + 로그아웃 버튼**이 항상 보입니다.
- **QR 방문 인증 → 리뷰 → 개선사항 → 상인 처리 → 관리자 이행관리**까지 이어지는 핵심 루프(FR-041~047)를 실제로 클릭해서 체험할 수 있습니다.

## 프로토타입 한계 (실서비스 전환 시 교체 필요)
- 카카오/네이버/구글 로그인은 실제 OAuth 대신 이메일 입력으로 시뮬레이션합니다.
- QR 스캔은 카메라 대신 "매장 선택 → 방문 인증 버튼"으로 시뮬레이션합니다.
- 관리자 2단계 인증(OTP)은 데모 버튼으로 대체했습니다.


## 1. 패키지 설치

In [ ]:
!pip install -q gradio


## 2. DB 스키마 + CSV 자동 적재 (`app_core`)
ERD/스키마(schema.sql) 23개 테이블 중 2주 MVP 핵심 8개 테이블(USER·MARKET·MARKET_STALL·MERCHANT·QR_CODE·QR_VISIT_LOG·REVIEW·IMPROVEMENT) + ADMIN·FAVORITE·NOTICE를 포함합니다.
`전통시장_상가매칭_광주전남_FULL.csv`를 자동으로 읽어 MARKET/MARKET_STALL을 채웁니다.

In [ ]:
# ============================================================
# AI기반 지역관광 맞춤 여행 플래너 - 통합 프로토타입
# 사용자 / 상인 / 관리자 3분할 화면 + 전통시장 DB(SQLite) 연동
# PRD v18 FR-001~012, FR-041~050, FR-061 기준
# ============================================================

import sqlite3
import uuid
import hashlib
import csv
import os
from datetime import datetime

import gradio as gr

DB_PATH = "travel_planner.db"
CSV_PATH = "전통시장_상가매칭_광주전남_FULL.csv"

# ------------------------------------------------------------
# 0. DB 스키마 (schema.sql / ERD.md 확정본 그대로, IF NOT EXISTS)
# ------------------------------------------------------------
SCHEMA_SQL = """
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS USER (
  user_id TEXT PRIMARY KEY,
  name TEXT NOT NULL,
  email TEXT,
  birthdate TEXT,
  auth_provider TEXT CHECK(auth_provider IN ('kakao','naver','google','pass','email')),
  auth_provider_id TEXT,
  identity_verified INTEGER DEFAULT 0,
  status TEXT DEFAULT 'active',
  created_at TEXT DEFAULT CURRENT_TIMESTAMP,
  travel_style TEXT
);

CREATE TABLE IF NOT EXISTS ADMIN (
  admin_id TEXT PRIMARY KEY,
  name TEXT NOT NULL,
  email TEXT,
  role TEXT,
  invited_by TEXT REFERENCES ADMIN(admin_id),
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS MARKET (
  market_id TEXT PRIMARY KEY,
  market_name TEXT NOT NULL,
  market_type TEXT,
  open_cycle TEXT,
  address_road TEXT,
  address_jibun TEXT,
  lat REAL,
  lon REAL,
  store_count INTEGER,
  item_categories TEXT,
  phone TEXT,
  data_ref_date TEXT,
  managed_by TEXT REFERENCES ADMIN(admin_id)
);

CREATE TABLE IF NOT EXISTS MERCHANT (
  merchant_id TEXT PRIMARY KEY,
  name TEXT NOT NULL,
  business_number TEXT,
  phone TEXT,
  email TEXT,
  auth_provider TEXT,
  password_hash TEXT,
  market_id TEXT REFERENCES MARKET(market_id),
  approval_status TEXT DEFAULT 'pending' CHECK(approval_status IN ('pending','approved','rejected')),
  approved_by TEXT REFERENCES ADMIN(admin_id),
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS MARKET_STALL (
  stall_id TEXT PRIMARY KEY,
  market_id TEXT NOT NULL REFERENCES MARKET(market_id),
  merchant_id TEXT REFERENCES MERCHANT(merchant_id),
  stall_number TEXT,
  pos_x REAL,
  pos_y REAL,
  stall_type TEXT,
  biz_name TEXT,
  category_large TEXT,
  category_mid TEXT,
  category_small TEXT,
  road_address TEXT,
  distance_m REAL,
  source_biz_id TEXT,
  data_source TEXT DEFAULT '소진공_상가업소_202603'
);

CREATE TABLE IF NOT EXISTS QR_CODE (
  qr_id TEXT PRIMARY KEY,
  stall_id TEXT NOT NULL REFERENCES MARKET_STALL(stall_id),
  qr_image_url TEXT,
  issued_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS QR_VISIT_LOG (
  visit_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  qr_id TEXT NOT NULL REFERENCES QR_CODE(qr_id),
  visited_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS REVIEW (
  review_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  target_type TEXT CHECK(target_type IN ('place','market','stall')),
  target_id TEXT,
  rating INTEGER CHECK(rating BETWEEN 1 AND 5),
  content TEXT,
  qr_verified INTEGER DEFAULT 0,
  source_visit_id TEXT REFERENCES QR_VISIT_LOG(visit_id),
  is_reported INTEGER DEFAULT 0,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS FAVORITE (
  favorite_id TEXT PRIMARY KEY,
  user_id TEXT NOT NULL REFERENCES USER(user_id),
  target_type TEXT,
  target_id TEXT,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS NOTICE (
  notice_id TEXT PRIMARY KEY,
  author_type TEXT CHECK(author_type IN ('merchant','admin')),
  author_id TEXT,
  market_id TEXT REFERENCES MARKET(market_id),
  title TEXT,
  content TEXT,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS IMPROVEMENT (
  improvement_id TEXT PRIMARY KEY,
  market_id TEXT NOT NULL REFERENCES MARKET(market_id),
  stall_id TEXT REFERENCES MARKET_STALL(stall_id),
  source_review_id TEXT REFERENCES REVIEW(review_id),
  content TEXT,
  status TEXT DEFAULT '접수' CHECK(status IN ('접수','처리중','완료')),
  handled_by TEXT REFERENCES ADMIN(admin_id),
  merchant_comment TEXT,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE INDEX IF NOT EXISTS idx_stall_market ON MARKET_STALL(market_id);
CREATE INDEX IF NOT EXISTS idx_stall_merchant ON MARKET_STALL(merchant_id);
CREATE INDEX IF NOT EXISTS idx_qrcode_stall ON QR_CODE(stall_id);
CREATE INDEX IF NOT EXISTS idx_review_target ON REVIEW(target_type, target_id);
CREATE INDEX IF NOT EXISTS idx_market_geo ON MARKET(lat, lon);
"""


def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def init_schema():
    conn = sqlite3.connect(DB_PATH)
    conn.executescript(SCHEMA_SQL)
    conn.commit()
    conn.close()


def hash_pw(pw: str) -> str:
    return hashlib.sha256(pw.encode("utf-8")).hexdigest()


# ------------------------------------------------------------
# 1. CSV -> MARKET / MARKET_STALL 자동 적재 (기존 로직 재사용)
# ------------------------------------------------------------
def load_data_from_csv(csv_path=CSV_PATH, force=False):
    if not os.path.exists(csv_path):
        print(f"'{csv_path}' 파일이 없어 CSV 적재를 건너뜁니다. 좌측 파일 탭에 업로드했는지 확인하세요.")
        return

    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM MARKET_STALL")
    if cur.fetchone()[0] > 0 and not force:
        print("MARKET_STALL에 이미 데이터가 있어 CSV 적재를 건너뜁니다.")
        conn.close()
        return

    with open(csv_path, encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))

    markets = {}
    for row in rows:
        name = row["시장명"]
        m = markets.setdefault(name, {
            "시장유형": row["시장유형"], "시장개설주기": row["시장개설주기"],
            "취급품목": row["취급품목"], "lats": [], "lons": [],
            "addr_candidates": [], "store_count": 0,
        })
        m["store_count"] += 1
        try:
            m["lats"].append(float(row["위도"])); m["lons"].append(float(row["경도"]))
        except (ValueError, KeyError):
            pass
        try:
            m["addr_candidates"].append((float(row["거리(m)"]), row["도로명주소"]))
        except (ValueError, KeyError):
            pass

    name_to_id = {}
    for name, m in markets.items():
        market_id = str(uuid.uuid4())
        name_to_id[name] = market_id
        lat = sum(m["lats"]) / len(m["lats"]) if m["lats"] else None
        lon = sum(m["lons"]) / len(m["lons"]) if m["lons"] else None
        addr = min(m["addr_candidates"])[1] if m["addr_candidates"] else None
        cur.execute(
            """INSERT INTO MARKET (market_id, market_name, market_type, open_cycle,
                                    address_road, lat, lon, store_count, item_categories, data_ref_date)
               VALUES (?,?,?,?,?,?,?,?,?,?)""",
            (market_id, name, m["시장유형"], m["시장개설주기"], addr, lat, lon,
             m["store_count"], m["취급품목"], "202603"),
        )

    inserted = 0
    for row in rows:
        market_id = name_to_id.get(row["시장명"])
        if not market_id:
            continue
        cur.execute(
            """INSERT INTO MARKET_STALL
               (stall_id, market_id, stall_type, biz_name, category_large, category_mid,
                category_small, road_address, distance_m, source_biz_id)
               VALUES (?,?,?,?,?,?,?,?,?,?)""",
            (str(uuid.uuid4()), market_id, row["시장유형"], row["상호명"],
             row["업종대분류"], row["업종중분류"], row["업종소분류"], row["도로명주소"],
             float(row["거리(m)"]) if row["거리(m)"] else None, row["상가업소번호"]),
        )
        inserted += 1

    conn.commit()
    conn.close()
    print(f"MARKET {len(markets)}건, MARKET_STALL {inserted}건 적재 완료")


TRAVEL_STYLE_OPTIONS = ["자연/힐링", "맛집/미식", "전통시장", "문화/역사", "액티비티/체험", "카페/디저트", "사진/인생샷"]
PROVIDER_LABEL = {"kakao": "카카오", "naver": "네이버", "google": "구글", "email": "이메일(자체)"}


# ------------------------------------------------------------
# 2. 사용자(USER) 인증 - FR-001~007
# ------------------------------------------------------------
def signup_user(provider, name, email, birthdate, password, password_confirm, travel_styles):
    if not name or not email:
        return "❌ 이름과 이메일은 필수입니다."
    if provider == "email":
        if not password:
            return "❌ 비밀번호를 입력해주세요."
        if len(password) < 8:
            return "❌ 비밀번호는 8자 이상이어야 합니다. (FR-005)"
        if password != password_confirm:
            return "❌ 비밀번호와 비밀번호 확인이 일치하지 않습니다."

    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT auth_provider FROM USER WHERE email = ?", (email,))
    existing = cur.fetchone()
    if existing:
        conn.close()
        if existing[0] != provider:
            return f"❌ 이미 '{PROVIDER_LABEL[existing[0]]}'로 가입된 이메일입니다. 해당 방법으로 로그인해주세요."
        return "❌ 이미 가입된 이메일입니다."

    user_id = str(uuid.uuid4())
    style_str = ",".join(travel_styles) if travel_styles else None
    pw_field = hash_pw(password) if provider == "email" else None
    cur.execute(
        """INSERT INTO USER (user_id, name, email, birthdate, auth_provider, auth_provider_id,
                              identity_verified, status, created_at, travel_style)
           VALUES (?,?,?,?,?,?,0,'active',?,?)""",
        (user_id, name, email, birthdate, provider, pw_field, datetime.now().isoformat(), style_str),
    )
    conn.commit(); conn.close()
    return f"✅ {PROVIDER_LABEL[provider]} 회원가입 완료! (온보딩 완료) 로그인 탭에서 로그인해주세요."


def login_user(provider, email, password):
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT user_id, name, auth_provider, auth_provider_id FROM USER WHERE email = ?", (email,))
    row = cur.fetchone(); conn.close()
    if not row:
        return "❌ 등록되지 않은 이메일입니다.", None, None
    user_id, name, real_provider, pw_hash = row
    if real_provider != provider:
        return f"❌ 이 계정은 '{PROVIDER_LABEL[real_provider]}'로 가입되어 있습니다.", None, None
    if provider == "email" and pw_hash != hash_pw(password):
        return "❌ 로그인 정보가 올바르지 않습니다.", None, None
    return f"✅ {name}님, 환영합니다! (일반 사용자)", user_id, "user"


# ------------------------------------------------------------
# 3. 상인(MERCHANT) 인증 - FR-008,009
# ------------------------------------------------------------
def get_market_name_list():
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT market_name FROM MARKET ORDER BY market_name")
    rows = [r[0] for r in cur.fetchall()]
    conn.close()
    return rows


def signup_merchant(name, business_number, phone, email, password, password_confirm, market_name):
    if not name or not business_number or not email or not market_name:
        return "❌ 이름/사업자등록번호/이메일/입점 시장은 필수입니다."
    if len(business_number.replace("-", "")) != 10 or not business_number.replace("-", "").isdigit():
        return "❌ 사업자등록번호 형식이 올바르지 않습니다. (숫자 10자리)"
    if len(password) < 8 or password != password_confirm:
        return "❌ 비밀번호는 8자 이상이어야 하며, 확인란과 일치해야 합니다."

    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT merchant_id FROM MERCHANT WHERE business_number=? OR email=?", (business_number, email))
    if cur.fetchone():
        conn.close()
        return "❌ 이미 가입된 사업자등록번호 또는 이메일입니다."

    cur.execute("SELECT market_id FROM MARKET WHERE market_name=?", (market_name,))
    m = cur.fetchone()
    if not m:
        conn.close()
        return "❌ 입점 시장 정보를 찾을 수 없습니다."
    market_id = m[0]

    merchant_id = str(uuid.uuid4())
    cur.execute(
        """INSERT INTO MERCHANT (merchant_id, name, business_number, phone, email, auth_provider,
                                  password_hash, market_id, approval_status, created_at)
           VALUES (?,?,?,?,?,'email',?,?,'pending',?)""",
        (merchant_id, name, business_number, phone, email, hash_pw(password), market_id, datetime.now().isoformat()),
    )
    conn.commit(); conn.close()
    return "✅ 입점 신청이 접수되었습니다. 관리자 승인 후 상인 대시보드를 이용할 수 있어요. (FR-009)"


def login_merchant(email, password):
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT merchant_id, name, password_hash, approval_status FROM MERCHANT WHERE email=?", (email,))
    row = cur.fetchone(); conn.close()
    if not row:
        return "❌ 등록되지 않은 상인 계정입니다. 입점 신청이 필요합니다. (FR-009)", None, None
    merchant_id, name, pw_hash, status = row
    if pw_hash != hash_pw(password):
        return "❌ 아이디 또는 비밀번호를 확인하세요.", None, None
    if status == "pending":
        return "⏳ 입점 심사중입니다. 관리자 승인 후 대시보드를 이용할 수 있어요.", None, None
    if status == "rejected":
        return "❌ 입점 신청이 반려되었습니다. 관리자에게 문의해주세요.", None, None
    return f"✅ {name}님, 환영합니다! (상인)", merchant_id, "merchant"


# ------------------------------------------------------------
# 4. 관리자(ADMIN) - FR-010,011 (셀프가입 없음, 초대기반)
#    프로토타입 한계: 실제 2단계 인증(OTP) 대신 데모 버튼으로 즉시 입장
# ------------------------------------------------------------
DEMO_ADMIN_EMAIL = "admin@demo.local"


def ensure_demo_admin():
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT admin_id FROM ADMIN WHERE email=?", (DEMO_ADMIN_EMAIL,))
    row = cur.fetchone()
    if row:
        admin_id = row[0]
    else:
        admin_id = str(uuid.uuid4())
        cur.execute(
            "INSERT INTO ADMIN (admin_id, name, email, role, created_at) VALUES (?,?,?,?,?)",
            (admin_id, "데모 최고관리자", DEMO_ADMIN_EMAIL, "최고관리자", datetime.now().isoformat()),
        )
        conn.commit()
    conn.close()
    return admin_id


def admin_quick_login():
    admin_id = ensure_demo_admin()
    msg = ("✅ 관리자 계정으로 바로 입장했습니다. (데모 전용 버튼)\n"
           "실서비스에서는 FR-010(2단계 인증)·FR-011(초대 기반 발급)로 대체됩니다.")
    return msg, admin_id, "admin"


# ------------------------------------------------------------
# 5. 로그아웃 + 화면 전환
# ------------------------------------------------------------
def logout():
    return "👋 로그아웃되었습니다.", None, None


def apply_role_visibility(role):
    """role: 'user' | 'merchant' | 'admin' | None(비로그인/게스트)
    반환 순서: [tab_user_chat, tab_user_market, tab_user_my, tab_merchant, tab_admin, logout_btn]"""
    is_user_or_guest = role in (None, "user")
    is_merchant = role == "merchant"
    is_admin = role == "admin"
    return (
        gr.update(visible=is_user_or_guest),
        gr.update(visible=is_user_or_guest),
        gr.update(visible=(role == "user")),
        gr.update(visible=is_merchant),
        gr.update(visible=is_admin),
        gr.update(visible=(role is not None)),
    )




## 3. 화면별 기능 로직 (`app_features`)
사용자(지역검색·시장상세·찜·QR인증리뷰) / 상인(내매장·QR발급·개선사항처리) / 관리자(입점승인·개선사항취합·공지)

In [ ]:
import uuid
from datetime import datetime

# ------------------------------------------------------------
# 사용자 - 지역 검색(FR-014) / 시장 상세(FR-015)
# ------------------------------------------------------------
def get_region_list():
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT DISTINCT address_road FROM MARKET")
    rows = cur.fetchall(); conn.close()
    regions = sorted({r[0].split()[0] for r in rows if r[0]})
    return ["전체"] + regions


def get_market_type_list():
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT DISTINCT market_type FROM MARKET")
    rows = cur.fetchall(); conn.close()
    return ["전체"] + sorted({r[0] for r in rows if r[0]})


def search_markets(region, market_type, keyword):
    conn = get_conn(); cur = conn.cursor()
    q = "SELECT market_name, market_type, open_cycle, address_road, store_count, item_categories FROM MARKET WHERE 1=1"
    params = []
    if region and region != "전체":
        q += " AND address_road LIKE ?"; params.append(f"{region}%")
    if market_type and market_type != "전체":
        q += " AND market_type LIKE ?"; params.append(f"%{market_type}%")
    if keyword:
        q += " AND market_name LIKE ?"; params.append(f"%{keyword}%")
    q += " ORDER BY store_count DESC"
    cur.execute(q, params); rows = cur.fetchall(); conn.close()
    if not rows:
        return [["검색 결과가 없습니다.", "", "", "", "", ""]]
    return [list(r) for r in rows]


def get_market_name_list_for_search():
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT market_name FROM MARKET ORDER BY market_name")
    rows = [r[0] for r in cur.fetchall()]
    conn.close()
    return rows


def get_market_detail(market_name):
    if not market_name:
        return "시장을 선택해주세요.", [], []
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT market_name, market_type, open_cycle, address_road, store_count, item_categories, phone
           FROM MARKET WHERE market_name=?""", (market_name,))
    market = cur.fetchone()
    if not market:
        conn.close()
        return "시장 정보를 찾을 수 없습니다.", [], []
    info_md = (f"### {market[0]}\n- **유형**: {market[1]}\n- **개설주기**: {market[2] or '상설(매일)'}\n"
               f"- **주소**: {market[3]}\n- **공식 점포수**: {market[4]}\n- **취급품목**: {market[5]}\n"
               f"- **전화번호**: {market[6] or '정보없음'}")
    cur.execute(
        "SELECT DISTINCT category_large FROM MARKET_STALL WHERE market_id=(SELECT market_id FROM MARKET WHERE market_name=?)",
        (market_name,))
    categories = ["전체"] + sorted({r[0] for r in cur.fetchall() if r[0]})
    cur.execute(
        """SELECT biz_name, category_large, category_small, road_address, distance_m
           FROM MARKET_STALL WHERE market_id=(SELECT market_id FROM MARKET WHERE market_name=?)
           ORDER BY distance_m ASC LIMIT 200""", (market_name,))
    stalls = cur.fetchall(); conn.close()
    return info_md, categories, [list(s) for s in stalls]


def filter_market_stalls(market_name, category):
    if not market_name:
        return []
    conn = get_conn(); cur = conn.cursor()
    q = """SELECT biz_name, category_large, category_small, road_address, distance_m
           FROM MARKET_STALL WHERE market_id=(SELECT market_id FROM MARKET WHERE market_name=?)"""
    params = [market_name]
    if category and category != "전체":
        q += " AND category_large=?"; params.append(category)
    q += " ORDER BY distance_m ASC LIMIT 200"
    cur.execute(q, params); rows = cur.fetchall(); conn.close()
    return [list(r) for r in rows]


# ------------------------------------------------------------
# 사용자 - 찜하기(FR-059)
# ------------------------------------------------------------
def add_favorite(user_id, market_name):
    if not user_id:
        return "❌ 먼저 로그인해주세요."
    if not market_name:
        return "❌ 찜할 시장을 선택해주세요."
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT market_id FROM MARKET WHERE market_name=?", (market_name,))
    m = cur.fetchone()
    if not m:
        conn.close(); return "❌ 시장 정보를 찾을 수 없습니다."
    market_id = m[0]
    cur.execute("SELECT favorite_id FROM FAVORITE WHERE user_id=? AND target_type='market' AND target_id=?",
                (user_id, market_id))
    if cur.fetchone():
        conn.close(); return "이미 찜한 시장입니다."
    cur.execute("INSERT INTO FAVORITE (favorite_id,user_id,target_type,target_id,created_at) VALUES (?,?,?,?,?)",
                (str(uuid.uuid4()), user_id, "market", market_id, datetime.now().isoformat()))
    conn.commit(); conn.close()
    return f"✅ '{market_name}'을(를) 찜 목록에 추가했습니다."


def get_my_favorites(user_id):
    if not user_id:
        return [["로그인이 필요합니다.", "", ""]]
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT m.market_name, m.market_type, m.address_road
           FROM FAVORITE f JOIN MARKET m ON f.target_id=m.market_id
           WHERE f.user_id=? AND f.target_type='market' ORDER BY f.created_at DESC""", (user_id,))
    rows = cur.fetchall(); conn.close()
    if not rows:
        return [["찜한 시장이 없습니다.", "", ""]]
    return [list(r) for r in rows]


# ------------------------------------------------------------
# 사용자 - QR 방문인증(FR-042) + 인증 리뷰 작성(FR-043)
# 프로토타입 한계: 실제 카메라 QR스캔 대신 "매장 선택 → 방문 인증" 버튼으로 시뮬레이션
# ------------------------------------------------------------
def get_stall_choices(market_name):
    if not market_name:
        return []
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT biz_name, stall_id FROM MARKET_STALL
           WHERE market_id=(SELECT market_id FROM MARKET WHERE market_name=?) LIMIT 300""", (market_name,))
    rows = cur.fetchall(); conn.close()
    return [(f"{name} ({sid[:8]})", sid) for name, sid in rows]


def qr_visit_and_review(user_id, stall_id, rating, content):
    if not user_id:
        return "❌ 먼저 로그인해주세요."
    if not stall_id:
        return "❌ 방문 인증할 매장을 선택해주세요."

    conn = get_conn(); cur = conn.cursor()
    # 1) 매장에 QR코드가 없으면 발급 (FR-041)
    cur.execute("SELECT qr_id FROM QR_CODE WHERE stall_id=?", (stall_id,))
    qr = cur.fetchone()
    if qr:
        qr_id = qr[0]
    else:
        qr_id = str(uuid.uuid4())
        cur.execute("INSERT INTO QR_CODE (qr_id, stall_id, issued_at) VALUES (?,?,?)",
                    (qr_id, stall_id, datetime.now().isoformat()))

    # 2) 방문 인증 로그 (FR-042)
    visit_id = str(uuid.uuid4())
    cur.execute("INSERT INTO QR_VISIT_LOG (visit_id, user_id, qr_id, visited_at) VALUES (?,?,?,?)",
                (visit_id, user_id, qr_id, datetime.now().isoformat()))

    # 3) 인증 리뷰 작성 (FR-043)
    review_id = str(uuid.uuid4())
    cur.execute(
        """INSERT INTO REVIEW (review_id,user_id,target_type,target_id,rating,content,
                                qr_verified,source_visit_id,created_at)
           VALUES (?,?,?,?,?,?,1,?,?)""",
        (review_id, user_id, "stall", stall_id, int(rating), content, visit_id, datetime.now().isoformat()))

    # 4) 리뷰에 개선요청 키워드가 있으면 IMPROVEMENT 자동 등록 (FR-044 연동, 간이 규칙)
    trigger_words = ["불편", "개선", "아쉬워", "아쉬운", "별로", "불친절", "위생"]
    improvement_note = ""
    if any(w in (content or "") for w in trigger_words):
        cur.execute("SELECT market_id FROM MARKET_STALL WHERE stall_id=?", (stall_id,))
        market_id = cur.fetchone()[0]
        imp_id = str(uuid.uuid4())
        cur.execute(
            """INSERT INTO IMPROVEMENT (improvement_id, market_id, stall_id, source_review_id, content, status, created_at)
               VALUES (?,?,?,?,?,'접수',?)""",
            (imp_id, market_id, stall_id, review_id, content, datetime.now().isoformat()))
        improvement_note = "\n🔔 리뷰 내용에서 개선 요청 사항이 감지되어 상인·관리자에게 자동 전달되었습니다. (FR-044~047)"

    conn.commit(); conn.close()
    return f"✅ QR 방문 인증 완료 + 인증 리뷰가 등록되었습니다. (qr_verified=1){improvement_note}"


# ------------------------------------------------------------
# 상인 - 내 정보 / QR / 리뷰·개선사항 조회·처리 (FR-041,044,045,048)
# ------------------------------------------------------------
def get_merchant_info(merchant_id):
    if not merchant_id:
        return "로그인이 필요합니다."
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT me.name, me.business_number, me.approval_status, ma.market_name
           FROM MERCHANT me LEFT JOIN MARKET ma ON me.market_id=ma.market_id
           WHERE me.merchant_id=?""", (merchant_id,))
    row = cur.fetchone(); conn.close()
    if not row:
        return "상인 정보를 찾을 수 없습니다."
    name, biz_no, status, market_name = row
    status_label = {"approved": "✅ 승인완료", "pending": "⏳ 심사중", "rejected": "❌ 반려"}.get(status, status)
    return f"### {name}님\n- 사업자등록번호: {biz_no}\n- 입점 시장: {market_name}\n- 승인 상태: {status_label}"


def get_my_stalls(merchant_id):
    """상인이 승인되면 본인 매장(구역)을 직접 등록해 QR을 발급받는 화면 (FR-041)"""
    if not merchant_id:
        return [["로그인이 필요합니다.", ""]]
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT s.biz_name, q.qr_id FROM MARKET_STALL s LEFT JOIN QR_CODE q ON s.stall_id=q.stall_id
           WHERE s.merchant_id=?""", (merchant_id,))
    rows = cur.fetchall(); conn.close()
    if not rows:
        return [["등록된 매장이 없습니다. 아래에서 매장을 등록해주세요.", ""]]
    return [[name, "발급됨" if qr else "미발급"] for name, qr in rows]


def register_my_stall(merchant_id, stall_number):
    if not merchant_id:
        return "❌ 로그인이 필요합니다.", get_my_stalls(merchant_id)
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT market_id, name, approval_status FROM MERCHANT WHERE merchant_id=?", (merchant_id,))
    row = cur.fetchone()
    if not row or row[2] != "approved":
        conn.close()
        return "❌ 승인된 상인만 매장을 등록할 수 있습니다.", get_my_stalls(merchant_id)
    market_id, biz_name, _ = row
    stall_id = str(uuid.uuid4())
    cur.execute(
        """INSERT INTO MARKET_STALL (stall_id, market_id, merchant_id, stall_number, stall_type, biz_name)
           VALUES (?,?,?,?,?,?)""", (stall_id, market_id, merchant_id, stall_number or "-", "상설", biz_name))
    qr_id = str(uuid.uuid4())
    cur.execute("INSERT INTO QR_CODE (qr_id, stall_id, issued_at) VALUES (?,?,?)",
                (qr_id, stall_id, datetime.now().isoformat()))
    conn.commit(); conn.close()
    return f"✅ 매장 등록 + QR코드 자동 발급 완료 (qr_id: {qr_id[:8]}...) (FR-041)", get_my_stalls(merchant_id)


def get_my_reviews_and_improvements(merchant_id):
    if not merchant_id:
        return [["로그인이 필요합니다.", "", "", "", ""]]
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT r.rating, r.content, r.qr_verified, i.status, i.improvement_id
           FROM MARKET_STALL s
           JOIN REVIEW r ON r.target_type='stall' AND r.target_id=s.stall_id
           LEFT JOIN IMPROVEMENT i ON i.source_review_id=r.review_id
           WHERE s.merchant_id=? ORDER BY r.created_at DESC""", (merchant_id,))
    rows = cur.fetchall(); conn.close()
    if not rows:
        return [["아직 등록된 QR 인증 리뷰가 없습니다.", "", "", "", ""]]
    return [[r[0], r[1], "인증" if r[2] else "미인증", r[3] or "-", (r[4] or "")[:8]] for r in rows]


def update_improvement_status(merchant_id, improvement_id_prefix, new_status, comment):
    if not merchant_id or not improvement_id_prefix:
        return "❌ 처리할 개선사항을 선택해주세요.", get_my_reviews_and_improvements(merchant_id)
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT improvement_id FROM IMPROVEMENT WHERE improvement_id LIKE ?",
                (f"{improvement_id_prefix}%",))
    row = cur.fetchone()
    if not row:
        conn.close()
        return "❌ 개선사항을 찾을 수 없습니다.", get_my_reviews_and_improvements(merchant_id)
    cur.execute("UPDATE IMPROVEMENT SET status=?, merchant_comment=? WHERE improvement_id=?",
                (new_status, comment, row[0]))
    conn.commit(); conn.close()
    return "✅ 처리 상태가 저장되었습니다. (FR-045)", get_my_reviews_and_improvements(merchant_id)


# ------------------------------------------------------------
# 관리자 - 상인 승인/반려 (FR-009 연계), 개선사항 취합(FR-046,047), 공지(FR-061)
# ------------------------------------------------------------
def get_pending_merchants():
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT me.name, me.business_number, ma.market_name, me.merchant_id
           FROM MERCHANT me LEFT JOIN MARKET ma ON me.market_id=ma.market_id
           WHERE me.approval_status='pending'""")
    rows = cur.fetchall(); conn.close()
    if not rows:
        return [["대기중인 입점 신청이 없습니다.", "", "", ""]]
    return [[n, b, m, mid[:8]] for n, b, m, mid in rows]


def approve_merchant(admin_id, merchant_id_prefix, decision):
    if not admin_id:
        return "❌ 관리자 로그인이 필요합니다.", get_pending_merchants()
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT merchant_id FROM MERCHANT WHERE merchant_id LIKE ?", (f"{merchant_id_prefix}%",))
    row = cur.fetchone()
    if not row:
        conn.close()
        return "❌ 대상 상인을 찾을 수 없습니다.", get_pending_merchants()
    status = "approved" if decision == "승인" else "rejected"
    cur.execute("UPDATE MERCHANT SET approval_status=?, approved_by=? WHERE merchant_id=?",
                (status, admin_id, row[0]))
    conn.commit(); conn.close()
    return f"✅ 처리 완료: {decision}", get_pending_merchants()


def get_improvements_by_market():
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT ma.market_name, s.biz_name, i.content, i.status, i.improvement_id
           FROM IMPROVEMENT i
           JOIN MARKET ma ON i.market_id=ma.market_id
           LEFT JOIN MARKET_STALL s ON i.stall_id=s.stall_id
           ORDER BY i.created_at DESC""")
    rows = cur.fetchall(); conn.close()
    if not rows:
        return [["취합된 개선사항이 없습니다.", "", "", "", ""]]
    return [[mk, sn or "-", ct, st, iid[:8]] for mk, sn, ct, st, iid in rows]


def admin_update_improvement(admin_id, improvement_id_prefix, new_status):
    if not admin_id:
        return "❌ 관리자 로그인이 필요합니다.", get_improvements_by_market()
    conn = get_conn(); cur = conn.cursor()
    cur.execute("SELECT improvement_id FROM IMPROVEMENT WHERE improvement_id LIKE ?",
                (f"{improvement_id_prefix}%",))
    row = cur.fetchone()
    if not row:
        conn.close()
        return "❌ 대상을 찾을 수 없습니다.", get_improvements_by_market()
    cur.execute("UPDATE IMPROVEMENT SET status=?, handled_by=? WHERE improvement_id=?",
                (new_status, admin_id, row[0]))
    conn.commit(); conn.close()
    return "✅ 이행 관리 상태가 업데이트되었습니다. (FR-047)", get_improvements_by_market()


def post_notice(admin_id, market_name, title, content):
    if not admin_id:
        return "❌ 관리자 로그인이 필요합니다."
    if not title or not content:
        return "❌ 제목과 내용을 입력해주세요."
    conn = get_conn(); cur = conn.cursor()
    market_id = None
    if market_name and market_name != "전체":
        cur.execute("SELECT market_id FROM MARKET WHERE market_name=?", (market_name,))
        m = cur.fetchone()
        market_id = m[0] if m else None
    cur.execute(
        "INSERT INTO NOTICE (notice_id, author_type, author_id, market_id, title, content, created_at) VALUES (?,?,?,?,?,?,?)",
        (str(uuid.uuid4()), "admin", admin_id, market_id, title, content, datetime.now().isoformat()))
    conn.commit(); conn.close()
    return "✅ 공지사항이 등록되었습니다. (FR-061)"


def get_notices_for_merchant():
    conn = get_conn(); cur = conn.cursor()
    cur.execute(
        """SELECT n.title, n.content, ma.market_name, n.created_at
           FROM NOTICE n LEFT JOIN MARKET ma ON n.market_id=ma.market_id
           ORDER BY n.created_at DESC LIMIT 50""")
    rows = cur.fetchall(); conn.close()
    if not rows:
        return [["등록된 공지사항이 없습니다.", "", "", ""]]
    return [[t, c, m or "전체", d[:10]] for t, c, m, d in rows]




## 4. Gradio UI — 로그인 상태에 따라 탭이 자동으로 열리고 닫힙니다

In [ ]:
import gradio as gr

init_schema()
load_data_from_csv()

CSS = """
.gradio-container {max-width: 1180px !important;}
#role-badge {font-weight:700; font-size:15px;}
"""


def create_app() -> gr.Blocks:
    with gr.Blocks(title="AI기반 지역관광 맞춤 여행 플래너") as demo:
        role_state = gr.State(None)   # 'user' | 'merchant' | 'admin' | None
        id_state = gr.State(None)     # user_id / merchant_id / admin_id

        gr.Markdown("# 🏮 AI기반 지역관광 맞춤 여행 플래너 (광주·전남 전통시장 특화)")
        with gr.Row():
            role_badge = gr.Markdown("**현재 상태: 비로그인(게스트)**", elem_id="role-badge")
            logout_btn = gr.Button("🚪 로그아웃", visible=False, scale=0)

        with gr.Tabs():
            # ------------------------------------------------
            # 0. 로그인 / 회원가입 (항상 노출)
            # ------------------------------------------------
            with gr.Tab("🔐 로그인/회원가입"):
                with gr.Tabs():
                    with gr.Tab("사용자 회원가입 (FR-001~007)"):
                        su_provider = gr.Radio(
                            choices=[("카카오", "kakao"), ("네이버", "naver"), ("구글", "google"), ("자체(이메일)", "email")],
                            value="email", label="가입 방식")
                        gr.Markdown("_프로토타입 한계: 카카오/네이버/구글은 실제 OAuth 대신 이메일 입력만으로 시뮬레이션합니다._")
                        su_name = gr.Textbox(label="이름")
                        su_email = gr.Textbox(label="이메일")
                        su_birth = gr.Textbox(label="생년월일 (YYYY-MM-DD)")
                        with gr.Group() as su_pw_group:
                            su_pw = gr.Textbox(label="비밀번호 (8자 이상)", type="password")
                            su_pw_confirm = gr.Textbox(label="비밀번호 확인", type="password")
                        su_style = gr.CheckboxGroup(choices=TRAVEL_STYLE_OPTIONS, label="선호 여행 스타일 (다중 선택, FR-007)")
                        su_btn = gr.Button("회원가입", variant="primary")
                        su_result = gr.Markdown()
                        su_provider.change(lambda p: gr.update(visible=(p == "email")), su_provider, su_pw_group)
                        su_btn.click(signup_user, [su_provider, su_name, su_email, su_birth, su_pw, su_pw_confirm, su_style], su_result)

                    with gr.Tab("사용자 로그인"):
                        li_provider = gr.Radio(
                            choices=[("카카오", "kakao"), ("네이버", "naver"), ("구글", "google"), ("자체(이메일)", "email")],
                            value="email", label="로그인 방식")
                        li_email = gr.Textbox(label="이메일")
                        li_pw = gr.Textbox(label="비밀번호 (자체 로그인만 해당)", type="password")
                        li_btn = gr.Button("로그인", variant="primary")
                        li_result = gr.Markdown()

                    with gr.Tab("상인 입점 신청 (FR-009)"):
                        gr.Markdown("사업자등록번호 등 정보를 입력해 입점을 신청합니다. **관리자 승인 후** 상인 대시보드를 이용할 수 있어요.")
                        mm_name = gr.Textbox(label="대표자명")
                        mm_biz = gr.Textbox(label="사업자등록번호 (숫자 10자리)")
                        mm_phone = gr.Textbox(label="연락처")
                        mm_email = gr.Textbox(label="이메일 (로그인 아이디)")
                        mm_pw = gr.Textbox(label="비밀번호 (8자 이상)", type="password")
                        mm_pw_confirm = gr.Textbox(label="비밀번호 확인", type="password")
                        mm_market = gr.Dropdown(choices=get_market_name_list(), label="입점할 시장 선택")
                        mm_btn = gr.Button("입점 신청", variant="primary")
                        mm_result = gr.Markdown()
                        mm_btn.click(signup_merchant, [mm_name, mm_biz, mm_phone, mm_email, mm_pw, mm_pw_confirm, mm_market], mm_result)

                    with gr.Tab("상인 로그인 (FR-008)"):
                        ml_email = gr.Textbox(label="이메일")
                        ml_pw = gr.Textbox(label="비밀번호", type="password")
                        ml_btn = gr.Button("상인 로그인", variant="primary")
                        ml_result = gr.Markdown()

                    with gr.Tab("관리자 (FR-010,011)"):
                        gr.Markdown(
                            "관리자는 셀프 회원가입이 없고, 최고관리자가 초대 기반으로 계정을 발급합니다.\n\n"
                            "**프로토타입 데모용**: 아래 버튼으로 실제 2단계 인증 없이 바로 관리자 화면에 입장할 수 있습니다."
                        )
                        admin_btn = gr.Button("🔑 관리자 계정으로 바로 입장 (데모)", variant="primary")
                        admin_result = gr.Markdown()

            # ------------------------------------------------
            # 1. 사용자 - AI 여행 챗봇 (자리만 유지, 필요시 관광공사 API 연동 확장)
            # ------------------------------------------------
            with gr.Tab("💬 AI 여행 챗봇", visible=True) as tab_user_chat:
                gr.Markdown(
                    "여행 스타일 기반 코스 추천 챗봇 자리입니다. (FR-016~022)\n\n"
                    "지금은 아래 '전통시장 둘러보기'에서 실제 DB 데이터를 검색해보세요. "
                    "관광공사 Open API 연동은 별도 인증키가 필요해 확장 지점으로 남겨두었습니다."
                )

            # ------------------------------------------------
            # 2. 사용자 - 전통시장 둘러보기 (FR-014,015)
            # ------------------------------------------------
            with gr.Tab("🏪 전통시장 둘러보기", visible=True) as tab_user_market:
                gr.Markdown("### 조건으로 전통시장 찾기")
                with gr.Row():
                    region_dd = gr.Dropdown(choices=get_region_list(), value="전체", label="지역(시/도)")
                    type_dd = gr.Dropdown(choices=get_market_type_list(), value="전체", label="시장 유형")
                    keyword_tb = gr.Textbox(label="시장명 검색")
                search_btn = gr.Button("검색", variant="primary")
                result_table = gr.Dataframe(headers=["시장명", "유형", "개설주기", "주소", "점포수", "취급품목"], interactive=False)
                search_btn.click(search_markets, [region_dd, type_dd, keyword_tb], result_table)

                gr.Markdown("### 시장 상세 · 찜하기")
                market_dd = gr.Dropdown(choices=get_market_name_list(), label="시장 선택")
                detail_md = gr.Markdown()
                category_dd = gr.Dropdown(choices=["전체"], value="전체", label="업종 필터")
                stall_table = gr.Dataframe(headers=["상호명", "업종대분류", "업종소분류", "도로명주소", "거리(m)"], interactive=False)
                fav_btn = gr.Button("❤️ 이 시장 찜하기")
                fav_result = gr.Markdown()
                market_dd.change(get_market_detail, market_dd, [detail_md, category_dd, stall_table])
                category_dd.change(filter_market_stalls, [market_dd, category_dd], stall_table)
                fav_btn.click(add_favorite, [id_state, market_dd], fav_result)

                gr.Markdown("### QR 방문 인증 + 리뷰 작성 (FR-041~044)")
                gr.Markdown("_실제 카메라 스캔 대신, 매장을 선택하고 '방문 인증' 버튼을 누르면 QR 스캔을 시뮬레이션합니다._")
                stall_dd = gr.Dropdown(choices=[], label="방문 인증할 매장 선택")
                market_dd.change(get_stall_choices, market_dd, stall_dd)
                qr_rating = gr.Slider(1, 5, value=5, step=1, label="별점")
                qr_content = gr.Textbox(label="리뷰 내용", placeholder="예: 친절하고 신선해요 / 위생이 조금 아쉬워요")
                qr_btn = gr.Button("📷 QR 방문 인증 + 리뷰 등록", variant="primary")
                qr_result = gr.Markdown()
                qr_btn.click(qr_visit_and_review, [id_state, stall_dd, qr_rating, qr_content], qr_result)

            # ------------------------------------------------
            # 3. 사용자 - 마이페이지 (FR-012, FR-059)
            # ------------------------------------------------
            with gr.Tab("🙋 마이페이지", visible=False) as tab_user_my:
                gr.Markdown("### 내 찜 목록")
                my_fav_btn = gr.Button("새로고침")
                my_fav_table = gr.Dataframe(headers=["시장명", "유형", "주소"], interactive=False)
                my_fav_btn.click(get_my_favorites, id_state, my_fav_table)

            # ------------------------------------------------
            # 4. 상인 대시보드 (FR-008,041,044,045,048)
            # ------------------------------------------------
            with gr.Tab("🧑‍🌾 상인 대시보드", visible=False) as tab_merchant:
                m_info_md = gr.Markdown()
                m_refresh_btn = gr.Button("내 정보 새로고침")
                m_refresh_btn.click(get_merchant_info, id_state, m_info_md)

                gr.Markdown("### 내 매장 · QR코드 (FR-041)")
                m_stall_table = gr.Dataframe(headers=["매장명", "QR 발급여부"], interactive=False)
                m_stall_number = gr.Textbox(label="새 매장(구역) 번호", placeholder="예: A-12")
                m_register_btn = gr.Button("매장 등록 + QR 자동발급", variant="primary")
                m_register_result = gr.Markdown()
                m_register_btn.click(register_my_stall, [id_state, m_stall_number], [m_register_result, m_stall_table])

                gr.Markdown("### 내 매장 QR 인증 리뷰 · 개선사항 (FR-044,045)")
                m_review_table = gr.Dataframe(
                    headers=["별점", "리뷰내용", "QR인증여부", "개선사항 상태", "개선사항ID(앞8자리)"], interactive=False)
                m_review_refresh = gr.Button("새로고침")
                m_review_refresh.click(get_my_reviews_and_improvements, id_state, m_review_table)

                with gr.Row():
                    m_imp_id = gr.Textbox(label="처리할 개선사항 ID (앞 8자리)")
                    m_imp_status = gr.Radio(choices=["접수", "처리중", "완료"], value="처리중", label="처리 상태")
                m_imp_comment = gr.Textbox(label="상인 코멘트")
                m_imp_btn = gr.Button("처리 상태 저장", variant="primary")
                m_imp_result = gr.Markdown()
                m_imp_btn.click(update_improvement_status, [id_state, m_imp_id, m_imp_status, m_imp_comment], [m_imp_result, m_review_table])

                gr.Markdown("### 공지사항 확인")
                m_notice_table = gr.Dataframe(headers=["제목", "내용", "대상 시장", "등록일"], interactive=False)
                m_notice_refresh = gr.Button("공지 새로고침")
                m_notice_refresh.click(get_notices_for_merchant, None, m_notice_table)

            # ------------------------------------------------
            # 5. 관리자 대시보드 (FR-009 승인,046,047,050,061)
            # ------------------------------------------------
            with gr.Tab("🛠️ 관리자 대시보드", visible=False) as tab_admin:
                gr.Markdown("### 상인 입점 승인 관리 (FR-009 연계)")
                a_pending_table = gr.Dataframe(headers=["대표자명", "사업자번호", "입점시장", "상인ID(앞8자리)"], interactive=False)
                a_pending_refresh = gr.Button("새로고침")
                a_pending_refresh.click(get_pending_merchants, None, a_pending_table)
                with gr.Row():
                    a_merchant_id = gr.Textbox(label="처리할 상인 ID (앞 8자리)")
                    a_decision = gr.Radio(choices=["승인", "반려"], value="승인", label="결정")
                a_approve_btn = gr.Button("승인/반려 처리", variant="primary")
                a_approve_result = gr.Markdown()
                a_approve_btn.click(approve_merchant, [id_state, a_merchant_id, a_decision], [a_approve_result, a_pending_table])

                gr.Markdown("### 시장별 개선사항 취합 (FR-046,047)")
                a_imp_table = gr.Dataframe(headers=["시장명", "매장명", "내용", "상태", "개선사항ID(앞8자리)"], interactive=False)
                a_imp_refresh = gr.Button("새로고침")
                a_imp_refresh.click(get_improvements_by_market, None, a_imp_table)
                with gr.Row():
                    a_imp_id = gr.Textbox(label="처리할 개선사항 ID (앞 8자리)")
                    a_imp_status = gr.Radio(choices=["접수", "처리중", "완료"], value="처리중", label="이행 상태")
                a_imp_btn = gr.Button("이행 상태 업데이트", variant="primary")
                a_imp_result = gr.Markdown()
                a_imp_btn.click(admin_update_improvement, [id_state, a_imp_id, a_imp_status], [a_imp_result, a_imp_table])

                gr.Markdown("### 공지사항 관리 (FR-061)")
                a_notice_market = gr.Dropdown(choices=["전체"] + get_market_name_list(), value="전체", label="대상 시장")
                a_notice_title = gr.Textbox(label="제목")
                a_notice_content = gr.Textbox(label="내용", lines=3)
                a_notice_btn = gr.Button("공지 등록", variant="primary")
                a_notice_result = gr.Markdown()
                a_notice_btn.click(post_notice, [id_state, a_notice_market, a_notice_title, a_notice_content], a_notice_result)

        # --------------------------------------------------
        # 로그인/로그아웃 성공 시 -> role_state/id_state 갱신 -> 탭 노출 갱신
        # --------------------------------------------------
        visibility_outputs = [tab_user_chat, tab_user_market, tab_user_my, tab_merchant, tab_admin, logout_btn]

        def _badge(role):
            label = {"user": "일반 사용자", "merchant": "상인", "admin": "관리자"}.get(role, "비로그인(게스트)")
            return f"**현재 상태: {label}**"

        li_event = li_btn.click(login_user, [li_provider, li_email, li_pw], [li_result, id_state, role_state])
        li_event.then(apply_role_visibility, role_state, visibility_outputs)
        li_event.then(_badge, role_state, role_badge)

        ml_event = ml_btn.click(login_merchant, [ml_email, ml_pw], [ml_result, id_state, role_state])
        ml_event.then(apply_role_visibility, role_state, visibility_outputs)
        ml_event.then(_badge, role_state, role_badge)

        admin_event = admin_btn.click(admin_quick_login, None, [admin_result, id_state, role_state])
        admin_event.then(apply_role_visibility, role_state, visibility_outputs)
        admin_event.then(_badge, role_state, role_badge)

        logout_event = logout_btn.click(logout, None, [role_badge, id_state, role_state])
        logout_event.then(lambda: "**현재 상태: 비로그인(게스트)**", None, role_badge)
        logout_event.then(apply_role_visibility, role_state, visibility_outputs)

    return demo




## 5. 앱 실행
실행하면 `share=True`로 공개 링크가 생성됩니다. (Colab 세션 종료 시 링크도 만료)

**데모 진행 팁**
1. 처음 들어가면 `🔐 로그인/회원가입` 탭만 보입니다.
2. "관리자로 바로 입장" 버튼을 눌러 상인 입점 신청을 승인해보세요.
3. 사용자로 회원가입/로그인 후 `🏪 전통시장 둘러보기`에서 QR 방문 인증 + 리뷰를 남겨보세요 (리뷰에 '불편', '아쉬워요' 등이 들어가면 개선사항이 자동 접수됩니다).
4. 상인으로 로그인해 개선사항 처리 상태를 등록하고, 관리자 화면에서 이행 현황을 확인해보세요.

In [ ]:
demo = create_app()
demo.launch(share=True, debug=True, css=CSS)
